# 04 - Real loop analysis (CPU): losslessness, wall-clock, simulator vs real (PLAN T9)
Input: `data/real_loop/` (unzipped `real_loop.zip` from Kaggle notebook 03) and `data/traces_v2/timing.json` (cost model from notebook 01). Policy parameters are read from the real run's `run_info.json`, i.e. exactly what ran.
Outputs: `results/phase1/real_loop.csv`, `real_loop_lossless.csv`, `real_loop_cost_model.csv`, `sim_vs_real.csv`.

In [ ]:
import json, os
from pathlib import Path
import numpy as np, pandas as pd
from lgmtp.gpu_traces import read_jsonl, load_tokenizer
from lgmtp.traces import load_traces
from lgmtp.vocab import build_masks
from lgmtp.policies import TokenTable
from lgmtp.morph import PrefixBoundaryModel
from lgmtp.simulator import load_timing, make_verify, simulate, summarize
from lgmtp.tuning import Context, make_policy, cap_for
from lgmtp import analysis

RL = Path(os.environ.get('LGMTP_REAL', '../data/real_loop'))
TIMING = Path(os.environ.get('LGMTP_TIMING', '../data/traces_v2/timing.json'))
PREFIX = Path(os.environ.get('LGMTP_PREFIX', '../results/phase1/prefix_model.pkl.gz'))
RES = Path(os.environ.get('LGMTP_RES', '../results/phase1')); RES.mkdir(parents=True, exist_ok=True)
info = json.loads((RL / 'run_info.json').read_text(encoding='utf-8'))
results = read_jsonl(RL / 'loop_results.jsonl')
refs = {r['id']: r for r in read_jsonl(RL / 'greedy_unbatched.jsonl')}
traces, tmeta = load_traces(RL / 'traces_real100.npz')
T = load_timing(TIMING); C_RATIO = T['c']; VERIFY = make_verify(T)
print(len(results), 'decodes |', info['n_sent'], 'sentences x', info['repeats'], 'repeats | policies', info['policies'])

## 1. Losslessness (vs unbatched greedy target output)

In [ ]:
div = json.loads((RL / 'divergences.json').read_text())
ll = pd.DataFrame([dict(policy=r['policy'], rep=r['rep'], id=r['id'], lossless=r['lossless']) for r in results])
lt = ll.groupby('policy').lossless.agg(['sum', 'size']).rename(columns={'sum': 'identical', 'size': 'decodes'}).reset_index()
lt.to_csv(RES / 'real_loop_lossless.csv', index=False); print(lt.to_string())
if div:
    d = pd.DataFrame(div); d.to_csv(RES / 'real_loop_divergences.csv', index=False)
    print(d.to_string()); print('max |margin| at divergences:', d.margin.abs().max())
else:
    print('no divergences')

## 2. Wall-clock (median over repeats, warm-up excluded)

In [ ]:
real = analysis.real_loop_tables(results)
real.to_csv(RES / 'real_loop.csv', index=False)
real.round(4)

## 3. Cost model check: measured time per round vs `g*t_draft` and `verify(g)*t_target`

In [ ]:
cm = analysis.cost_model_table(results, T['t_draft'], T['t_target'], VERIFY)
cm.to_csv(RES / 'real_loop_cost_model.csv', index=False)
cm.round(3)

## 4. Simulator vs real
Simulator on teacher-forced traces of the same 100 unbatched greedy outputs, same params and cost model. Relative errors > 5% are flagged; explanations to consider: teacher-forcing approximation (stopping after a mismatch), lookahead cost (charged vs uncharged bounds), cache / Python overhead in the real loop, the end-of-sentence cap (simulator caps at n - p, the live loop at max_new - p), and `c` measured separately in notebook 01.

In [ ]:
tok = load_tokenizer('meta-llama/Llama-3.2-1B-Instruct')
masks = build_masks(tok, tmeta['eos_ids'])
ctx = Context(newword_mask=masks.starts_newword, prefix_model=PrefixBoundaryModel.load(PREFIX),
              table=TokenTable.from_tokenizer(tok, masks))
assert [t.id for t in traces] == list(refs)[:len(traces)] or set(t.id for t in traces) <= set(refs)
sim_rows = []
for fam in info['policies']:
    params = info['selected'].get(fam, {})
    pol = make_policy(fam, params, ctx); cap = cap_for(fam, params)
    ch = summarize(simulate(traces, pol, cap, C_RATIO, VERIFY, charge_lookahead=True, record_rounds=False))
    un = summarize(simulate(traces, pol, cap, C_RATIO, VERIFY, charge_lookahead=False, record_rounds=False))
    sim_rows.append(dict(policy=fam, tokens_per_verif=ch['tokens_per_verif'], mean_draft_len=ch['mean_draft_len'],
                         speedup_sim_charged=ch['speedup'], speedup_sim_uncharged=un['speedup']))
svr = analysis.sim_vs_real(real, sim_rows)
svr.to_csv(RES / 'sim_vs_real.csv', index=False)
cols = ['policy', 'tokens_per_verif_real', 'tokens_per_verif_sim', 'mean_draft_len_real', 'mean_draft_len_sim',
        'speedup_wall', 'speedup_sim_charged', 'speedup_sim_uncharged'] + [c for c in svr.columns if c.startswith('relerr_')] + ['flag_gt_5pct']
svr[cols].round(4)